# Module 4 — Take-Home Exercises: Evaluation & Observability

These exercises reinforce what you learned about LLM-as-Judge evaluation,
custom evaluators, and interpreting evaluation results.

**Exercise 1-2:** No GPU needed (analysis + LangSmith API calls only)
**Exercise 3:** No GPU needed (conceptual design)

**Prerequisites:** LangSmith API key + OpenAI API key (same as class)

---

## Exercise 1: Create a Custom Evaluator

In class, we used 3 evaluators: helpfulness, accuracy, and safety.
Now you'll create a **4th evaluator: completeness**.

A complete medical answer should:
- Answer ALL parts of the question (not just the first part)
- Include relevant context (causes, symptoms, treatments, prevention)
- Not trail off mid-sentence or end abruptly

**Tasks:**

1. Write the scoring prompt for a `completeness` evaluator
2. Define what scores 5, 3, and 1 look like (with examples)
3. Run it on the v2 benchmark results alongside the original 3 evaluators
4. Does the completeness score correlate with the helpfulness regression we saw?

In [ ]:
!pip install -q langsmith openai pandas

In [ ]:
import os
from openai import OpenAI

# Set your API keys
os.environ["LANGCHAIN_API_KEY"] = "lsv2_pt_..."  # Your LangSmith key
os.environ["OPENAI_API_KEY"] = "sk-..."           # Your OpenAI key

client = OpenAI()

In [ ]:
# TODO: Design the completeness evaluator prompt
# Model it on the helpfulness/accuracy/safety evaluators from class

COMPLETENESS_PROMPT = """
You are evaluating a healthcare assistant's response for COMPLETENESS.

A complete response:
[TODO: Define what makes a response complete]

Score on a scale of 0-5:
  5 = [TODO: Define — example: "Addresses every aspect of the question..."]
  4 = [TODO]
  3 = [TODO: Define — example: "Covers the main point but misses..."]
  2 = [TODO]
  1 = [TODO: Define — example: "Only addresses one small part..."]
  0 = [TODO]

Question: {question}
Answer: {answer}

Return ONLY a JSON object: {{"score": <0-5>, "reasoning": "<brief explanation>"}}
"""

def evaluate_completeness(question, answer):
    """Score a response for completeness using GPT-4o-mini."""
    prompt = COMPLETENESS_PROMPT.format(question=question, answer=answer)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    # TODO: Parse the JSON response and return normalized score (0-1)
    import json
    result = json.loads(response.choices[0].message.content)
    return result["score"] / 5.0, result["reasoning"]

In [ ]:
import json

# Load v2 benchmark results
with open("../module_2_colab_finetuning/results/benchmark_results_v2.json") as f:
    v2_results = json.load(f)

# TODO: Score both base and fine-tuned outputs for completeness
print(f"{'Question':<50} {'Base':>6} {'FT':>6} {'Delta':>7}")
print("-" * 75)

for r in v2_results[:5]:  # Start with 5 to save API calls
    # base_score, base_reason = evaluate_completeness(r["prompt"], r["base_output"])
    # ft_score, ft_reason = evaluate_completeness(r["prompt"], r["finetuned_output"])
    # delta = ft_score - base_score
    # print(f"{r['prompt'][:48]:<50} {base_score:>5.2f} {ft_score:>5.2f} {delta:>+6.2f}")
    pass

**Your Findings (fill in):**

1. Average completeness: Base = ___, Fine-tuned v2 = ___
2. Does completeness correlate with the helpfulness regression? (yes/no)
3. Why might completeness be a better metric than helpfulness for catching
   the "too concise" problem?

---

## Exercise 2: Cross-Version Analysis (No GPU)

You have real evaluation data from both v1 and v2. Use it to build
a comprehensive comparison.

**Tasks:**

1. Load both v1 and v2 CSV files from LangSmith
2. Build a per-question comparison table showing all 3 metrics for:
   base model, fine-tuned v1, fine-tuned v2
3. Identify: which questions did v2 fix that v1 broke?
4. Identify: which questions did BOTH versions struggle with?
5. If you could only deploy ONE version, which would you choose and why?

In [ ]:
import pandas as pd

# Load LangSmith CSV exports
v1_df = pd.read_csv("results/BaseModel-Vs-FineTuned-v1.csv")
v2_df = pd.read_csv("results/BaseModel-Vs-FineTuned-v2.csv")

print(f"v1 rows: {len(v1_df)}, v2 rows: {len(v2_df)}")
print(f"\nv1 columns: {list(v1_df.columns)}")

In [ ]:
# TODO: Separate base model and fine-tuned rows
# The CSV contains BOTH base and fine-tuned runs
# Look at the 'session_name' column to tell them apart

# v1_base = v1_df[v1_df["session_name"].str.contains("base")]
# v1_ft = v1_df[v1_df["session_name"].str.contains("finetuned")]

# TODO: Do the same for v2
# TODO: Extract the question text from the 'inputs' column (it's JSON)
# HINT: import json; json.loads(row["inputs"])["question"]

In [ ]:
# TODO: Build the comparison table
# For each of the 10 questions, show:
#   Question | Base (acc/help/safe) | FT-v1 (acc/help/safe) | FT-v2 (acc/help/safe)

# HINT: Merge the dataframes on the question text
# Then compute deltas: v1_delta = ft_v1 - base, v2_delta = ft_v2 - base

**Your Analysis (fill in):**

1. Questions v2 FIXED that v1 BROKE (improved over base in v2, regressed in v1):
   - ...
2. Questions BOTH versions struggled with:
   - ...
3. If deploying ONE version, I'd choose: (base / v1 / v2) because...
4. What does this tell you about the limits of fine-tuning on 2,000 examples?

---

## Exercise 3: Design an Evaluation Suite for a New Domain

You've been asked to fine-tune a model for a **legal Q&A chatbot** that helps
people understand their tenant rights.

**Tasks (conceptual — no code required):**

1. **Design 3 evaluators** (like our helpfulness/accuracy/safety).
   For each, write:
   - The evaluator name
   - What it measures (1 sentence)
   - What a score of 5 looks like
   - What a score of 1 looks like
   - An example question + good answer + bad answer

2. **Write 10 benchmark questions** for tenant rights.
   Cover diverse topics: lease terms, security deposits, eviction process,
   maintenance responsibilities, discrimination, etc.

3. **Identify risks**: What's the equivalent of "medical safety" for legal advice?
   What disclaimers should every response include?

**Your Design (fill in):**

### Evaluator 1: ___
- Measures: ...
- Score 5: ...
- Score 1: ...
- Example Q: ...
- Good A: ...
- Bad A: ...

### Evaluator 2: ___
- Measures: ...
- Score 5: ...
- Score 1: ...

### Evaluator 3: ___
- Measures: ...
- Score 5: ...
- Score 1: ...

### 10 Benchmark Questions
1. ...
2. ...
3. ...
4. ...
5. ...
6. ...
7. ...
8. ...
9. ...
10. ...

### Safety Equivalent
- The legal equivalent of "consult a healthcare professional" is: ...
- Every response should include: ...
- The biggest risk of bad legal advice is: ...

---

## Exercise 4: Judge Model Comparison (Optional, needs OpenAI API)

We used GPT-4o-mini as our judge. But does the judge model matter?

**Tasks:**

1. Re-run the accuracy evaluator on 5 v2 outputs using GPT-4o (not mini)
2. Compare scores: does the more expensive model agree with GPT-4o-mini?
3. Calculate the cost difference (GPT-4o is ~15× more expensive per token)
4. Is the accuracy improvement worth the cost for a healthcare application?

In [ ]:
import json
from openai import OpenAI

client = OpenAI()

ACCURACY_PROMPT = """
You are evaluating a healthcare assistant's response for MEDICAL ACCURACY.

Score on a scale of 0-5:
  5 = All medical facts are correct, current, and well-supported
  3 = Mostly accurate with minor errors or outdated information
  1 = Contains significant medical errors that could mislead
  0 = Completely inaccurate or fabricated medical information

Question: {question}
Answer: {answer}

Return ONLY a JSON object: {{"score": <0-5>, "reasoning": "<brief explanation>"}}
"""

def evaluate_accuracy(question, answer, model="gpt-4o-mini"):
    prompt = ACCURACY_PROMPT.format(question=question, answer=answer)
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    result = json.loads(response.choices[0].message.content)
    return result["score"] / 5.0, result["reasoning"]

# Load v2 results
with open("../module_2_colab_finetuning/results/benchmark_results_v2.json") as f:
    v2_results = json.load(f)

# TODO: Compare GPT-4o-mini vs GPT-4o on 5 questions
# for r in v2_results[:5]:
#     mini_score, mini_reason = evaluate_accuracy(r["prompt"], r["finetuned_output"], model="gpt-4o-mini")
#     full_score, full_reason = evaluate_accuracy(r["prompt"], r["finetuned_output"], model="gpt-4o")
#     print(f"Q: {r['prompt'][:50]}")
#     print(f"  4o-mini: {mini_score:.2f} — {mini_reason}")
#     print(f"  4o:      {full_score:.2f} — {full_reason}")
#     print()

**Your Findings (fill in):**

1. Agreement between GPT-4o-mini and GPT-4o: ___% of scores within 0.2
2. When they disagreed, which model was stricter?
3. Cost: 5 evaluations with 4o-mini = ~$___, with 4o = ~$___
4. For a healthcare application, is the more expensive judge worth it? Why?